In this notebook we implement pybdshadow for calculating building's shadowing in bologna's city for adressing a Urban Thermal Adaptability index 

In [1]:
!pip install pybdshadow


[notice] A new release of pip is available: 24.0 -> 25.3
[notice] To update, run: C:\Users\guada\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import geopandas as gpd
import pybdshadow
import matplotlib.pyplot as plt
import json
import time
import os
from tqdm import tqdm
from pyproj import CRS
import numpy as np

import warnings
warnings.filterwarnings("ignore", message="CRS not set for some of the concatenation inputs")
warnings.simplefilter(action='ignore', category=FutureWarning)


In [ ]:
trees_vectors_path = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-vectors-clean/'
shadows_folder_path = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/trees-shadows/'
os.makedirs(shadows_folder_path, exist_ok=True)

In [ ]:
# computing shadows 
# THIS BLOCK TAKES AROUND 1H20' TO FINISH. 
'''

date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

for j in tqdm(range(num_parts)):
    part_file = os.path.join(input_folder, f"part{j+1}_clean.geojson")
    if not os.path.exists(part_file):
        continue
        
    trees = gpd.read_file(part_file)
    trees = trees.to_crs(CRS.from_epsg(4326)) # pybdshadow needs EPSG:4326
    trees = pybdshadow.bd_preprocess(trees)

    # Divide each one in 5 parts for avoiding problems with RAM
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts):
        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(part, date, roof=False, include_building=True)
            print(f"Processing time part{j+1} subpart{k+1}: {round(time.time() - start, 2)} seconds")
            shadow_part.append(shadow_subpart)
        except Exception as e:
            print(f"Error in part{j+1} subpart{k+1}: {e}")
        time.sleep(10)
    
    # join results
    shadows = gpd.GeoDataFrame(pd.concat(shadow_part, ignore_index=True), crs=shadow_part[0].crs)
    # dissolve
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    output_path = os.path.join(output_folder, f"part{j}_shadow.geojson")
    #shadows.set_crs("EPSG:32632", inplace=True)
    shadows.to_file(output_path, driver="GeoJSON")
'''

In [ ]:
'''
date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

for j in tqdm(range(num_parts)):
    part_file = os.path.join(input_folder, f"part{j+1}_clean.geojson")
    if not os.path.exists(part_file):
        continue

    # ----------------------------------------------------------------------
    # READ + CHECK EMPTY
    # ----------------------------------------------------------------------
    trees = gpd.read_file(part_file)
    
    # Skip if empty or geometry missing
    if trees.empty or trees.geometry.is_empty.all() or trees.geometry.isna().all():
        print(f"part{j+1}: EMPTY after read -> skipped")
        continue

    # ----------------------------------------------------------------------
    # CRS
    # ----------------------------------------------------------------------
    trees = trees.to_crs(CRS.from_epsg(4326))  # pybdshadow needs EPSG:4326

    # ----------------------------------------------------------------------
    # PREPROCESS + CHECK EMPTY
    # ----------------------------------------------------------------------
    trees = pybdshadow.bd_preprocess(trees)

    if trees.empty or trees.geometry.is_empty.all() or trees.geometry.isna().all():
        print(f"part{j+1}: EMPTY after bd_preprocess -> skipped")
        continue

    # ----------------------------------------------------------------------
    # SPLIT
    # ----------------------------------------------------------------------
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts):

        # Skip empty subpart
        if part.empty or part.geometry.is_empty.all() or part.geometry.isna().all():
            print(f"part{j+1} subpart{k+1}: EMPTY -> skipped")
            continue

        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(
                part, date, roof=False, include_building=True
            )
            print(f"Processing time part{j+1} subpart{k+1}: {round(time.time() - start, 2)}s")
            shadow_part.append(shadow_subpart)
        except Exception as e:
            print(f"Error in part{j+1} subpart{k+1}: {e}")

        time.sleep(10)

    # ----------------------------------------------------------------------
    # NO SUBPARTS → SKIP
    # ----------------------------------------------------------------------
    if not shadow_part:
        print(f"part{j+1}: No shadow parts produced -> skipped")
        continue

    # ----------------------------------------------------------------------
    # MERGE RESULTS
    # ----------------------------------------------------------------------
    shadows = gpd.GeoDataFrame(pd.concat(shadow_part, ignore_index=True),
                               crs=shadow_part[0].crs)

    # Skip if shadows empty
    if shadows.empty:
        print(f"part{j+1}: EMPTY shadows -> skipped")
        continue

    # ----------------------------------------------------------------------
    # DISSOLVE / UNION
    # ----------------------------------------------------------------------
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)

    # Skip if union empty
    if shadows.empty:
        print(f"part{j+1}: EMPTY union -> skipped")
        continue

    # ----------------------------------------------------------------------
    # SAVE OUTPUT
    # ----------------------------------------------------------------------
    output_path = os.path.join(output_folder, f"part{j}_shadow.geojson")
    shadows.to_file(output_path, driver="GeoJSON")
    '''


In [ ]:
date = pd.to_datetime('2024-08-09 09:58:10').tz_localize('UTC')

# iterate through all vector files in folder -- cambiar nombres rutas despues -- 35' 1/30
vector_ext = (".geojson")
files = [f for f in os.listdir(input_folder) if f.endswith(vector_ext)] #unificar esta linea y la anterior despues

for idx, fname in enumerate(files, start=1):

    print(f"\n=== Processing file {idx}/{len(files)}: {fname} ===")

    part_file = os.path.join(input_folder, fname)
    trees = gpd.read_file(part_file)
    trees = trees.to_crs(CRS.from_epsg(4326))  # pybdshadow needs EPSG:4326
    trees = pybdshadow.bd_preprocess(trees)

    # SPLIT
    subparts = np.array_split(trees, 5)
    shadow_part = []

    for k, part in enumerate(subparts, start=1):

        try:
            start = time.time()
            shadow_subpart = pybdshadow.bdshadow_sunlight(
                part, date, roof=False, include_building=True
            )
            print(f"Processing time {fname} subpart{k}: {round(time.time() - start, 2)}s")
            shadow_part.append(shadow_subpart)

        except Exception as e:
            print(f"Error in {fname} subpart{k}: {e}")

        time.sleep(10)

    # MERGE RESULTS
    shadows = gpd.GeoDataFrame(
        pd.concat(shadow_part, ignore_index=True),
        crs=shadow_part[0].crs
    )

    # DISSOLVE / UNION
    union = shadows.union_all()
    shadows = gpd.GeoDataFrame(geometry=[union], crs=shadows.crs).explode(index_parts=False)


    # SAVE OUTPUT
    outname = fname.replace(".", "_shadow.")
    output_path = os.path.join(output_folder, outname)
    shadows.to_file(output_path, driver="GeoJSON")

    print(f"Saved: {outname}")



=== Processing file 1/30: part1_clean_part0.geojson ===
Processing time part1_clean_part0.geojson subpart1: 224.03s
Processing time part1_clean_part0.geojson subpart2: 261.58s
Processing time part1_clean_part0.geojson subpart3: 914.88s
Processing time part1_clean_part0.geojson subpart4: 127.96s
Processing time part1_clean_part0.geojson subpart5: 145.46s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Saved: part1_clean_part0_shadow.geojson

=== Processing file 2/30: part1_clean_part1.geojson ===
Processing time part1_clean_part1.geojson subpart1: 130.71s
Processing time part1_clean_part1.geojson subpart2: 114.29s
Processing time part1_clean_part1.geojson subpart3: 183.88s
Processing time part1_clean_part1.geojson subpart4: 259.52s
Processing time part1_clean_part1.geojson subpart5: 159.7s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Saved: part1_clean_part1_shadow.geojson

=== Processing file 3/30: part1_clean_part2.geojson ===
Processing time part1_clean_part2.geojson subpart1: 137.96s
Processing time part1_clean_part2.geojson subpart2: 245.43s
Processing time part1_clean_part2.geojson subpart3: 177.14s
Error in part1_clean_part2.geojson subpart4: Unable to allocate 8.22 GiB for an array with shape (1102637200,) and data type int64
Processing time part1_clean_part2.geojson subpart5: 447.41s


...

Saved: part3_clean_part1_shadow.geojson

=== Processing file 13/30: part3_clean_part2.geojson ===
Processing time part3_clean_part2.geojson subpart1: 49.43s
Processing time part3_clean_part2.geojson subpart2: 49.0s
Processing time part3_clean_part2.geojson subpart3: 56.72s
Processing time part3_clean_part2.geojson subpart4: 59.79s
Processing time part3_clean_part2.geojson subpart5: 66.16s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Saved: part3_clean_part2_shadow.geojson

=== Processing file 14/30: part3_clean_part3.geojson ===
Processing time part3_clean_part3.geojson subpart1: 74.67s
Processing time part3_clean_part3.geojson subpart2: 68.89s
Processing time part3_clean_part3.geojson subpart3: 71.74s
Processing time part3_clean_part3.geojson subpart4: 64.01s
Processing time part3_clean_part3.geojson subpart5: 66.35s


C:\Users\guada\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


Saved: part3_clean_part3_shadow.geojson

=== Processing file 15/30: part3_clean_part4.geojson ===
Processing time part3_clean_part4.geojson subpart1: 96.51s
Processing time part3_clean_part4.geojson subpart2: 153.64s
Processing time part3_clean_part4.geojson subpart3: 79.46s


In [ ]:
# join all the results in one dataframe
'''
total_shadows = []

for j in tqdm(range(num_parts)):
    
    part_file = os.path.join(shadows_folder_path, f"part{j}_shadow.geojson") #correjir a j++1 aca y anterior bloque
    if not os.path.exists(part_file):
        continue
    shadow_part = gpd.read_file(part_file)
    total_shadows.append(shadow_part)

    # Unir y guardar
    shadows = gpd.GeoDataFrame(pd.concat(total_shadows, ignore_index=True), crs=total_shadows[0].crs)


shadows.to_file('../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/shadows_trees_2024-08-09_095810', driver="GeoJSON")
'''

In [ ]:
# collect all shadow files in the folder
shadow_files = [f for f in os.listdir(shadows_folder_path) if f.endswith(".geojson")]

total_shadows = []

# iterate through all shadow files found
for fname in tqdm(shadow_files):

    part_file = os.path.join(shadows_folder_path, fname)
    if not os.path.exists(part_file):
        continue

    # read file
    shadow_part = gpd.read_file(part_file)
    total_shadows.append(shadow_part)

# merge all
shadows = gpd.GeoDataFrame(
    pd.concat(total_shadows, ignore_index=True),
    crs=total_shadows[0].crs
)

# save output
output_path = '../../../../datos-notebooks/2.global-data-bologna/2.5.MRT-prediction/2.5.1.shadows/shadows_trees_2024-08-09_095810.geojson'
shadows.to_file(output_path, driver="GeoJSON")

print(f"Saved merged shadows → {output_path}")
